# PJM Load Forecast — DA Cutoff Analysis

Evaluates how PJM load forecasts evolve leading up to the DA market window.
The last forecast before 9am EST (7am MST) is what a trader would use for DA bidding.
We compare this cutoff forecast to 12h/24h/48h earlier vintages to reveal revision direction and magnitude,
then connect to DA LMP outcomes.

## 1. Setup & Data Pull

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))
from src.utils.azure_postgresql import pull_from_db

INFO:root:CONFIG_DIR: c:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\backend\src


In [2]:
# Load the main cutoff analysis query
sql_path = Path.cwd() / "pjm_load_forecast_da_cutoff.sql"
query = sql_path.read_text()
df = pull_from_db(query=query)

print(f"{len(df):,} rows")
print(f"Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()}")
print(f"Regions: {sorted(df['region'].unique())}")
df.head(10)

768 rows
Date range: 2026-02-25 to 2026-03-04
Regions: ['MIDATL', 'RTO', 'SOUTH', 'WEST']


,forecast_date,hour_ending,region,cutoff_load_mw,cutoff_execution_dt,exec_dt_12h,load_mw_12h,exec_dt_24h,load_mw_24h,exec_dt_48h,load_mw_48h,delta_12h,delta_24h,delta_48h
0,2026-03-04,1,MIDATL,29224.0,2026-03-04 08:47:16,2026-03-03 20:47:09,28617.0,2026-03-03 08:17:32,28252.0,2026-03-02 08:47:09,28253.0,607.0,972.0,971.0
1,2026-03-04,2,MIDATL,28370.0,2026-03-04 08:47:16,2026-03-03 20:47:09,27769.0,2026-03-03 08:17:32,27439.0,2026-03-02 08:47:09,27416.0,601.0,931.0,954.0
2,2026-03-04,3,MIDATL,27817.0,2026-03-04 08:47:16,2026-03-03 20:47:09,27340.0,2026-03-03 08:17:32,27043.0,2026-03-02 08:47:09,27011.0,477.0,774.0,806.0
3,2026-03-04,4,MIDATL,27689.0,2026-03-04 08:47:16,2026-03-03 20:47:09,27234.0,2026-03-03 08:17:32,26979.0,2026-03-02 08:47:09,26931.0,455.0,710.0,758.0
4,2026-03-04,5,MIDATL,28248.0,2026-03-04 08:47:16,2026-03-03 20:47:09,27752.0,2026-03-03 08:17:32,27525.0,2026-03-02 08:47:09,27475.0,496.0,723.0,773.0
5,2026-03-04,6,MIDATL,29916.0,2026-03-04 08:47:16,2026-03-03 20:47:09,29368.0,2026-03-03 08:17:32,29223.0,2026-03-02 08:47:09,29160.0,548.0,693.0,756.0
6,2026-03-04,7,MIDATL,32407.0,2026-03-04 08:47:16,2026-03-03 20:47:09,31719.0,2026-03-03 08:17:32,31699.0,2026-03-02 08:47:09,31648.0,688.0,708.0,759.0
7,2026-03-04,8,MIDATL,33416.0,2026-03-04 08:47:16,2026-03-03 20:47:09,32858.0,2026-03-03 08:17:32,32840.0,2026-03-02 08:47:09,32844.0,558.0,576.0,572.0
8,2026-03-04,9,MIDATL,33465.0,2026-03-04 08:47:16,2026-03-03 20:47:09,32800.0,2026-03-03 08:17:32,32553.0,2026-03-02 08:47:09,32630.0,665.0,912.0,835.0
9,2026-03-04,10,MIDATL,32825.0,2026-03-04 08:47:16,2026-03-03 20:47:09,32284.0,2026-03-03 08:17:32,31824.0,2026-03-02 08:47:09,31935.0,541.0,1001.0,890.0


In [3]:
df.dtypes

forecast_date                  object
hour_ending                     int64
region                            str
cutoff_load_mw                float64
cutoff_execution_dt    datetime64[us]
exec_dt_12h            datetime64[us]
load_mw_12h                   float64
exec_dt_24h            datetime64[us]
load_mw_24h                   float64
exec_dt_48h            datetime64[us]
load_mw_48h                   float64
delta_12h                     float64
delta_24h                     float64
delta_48h                     float64
dtype: object

## 2. Data Validation — Cutoff Vintage Inspection

In [4]:
# Actual cutoff execution timestamps per forecast_date
cutoff_times = (
    df.groupby("forecast_date")["cutoff_execution_dt"]
    .first()
    .reset_index()
)
cutoff_times["cutoff_hour"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.hour
cutoff_times["cutoff_minute"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.minute
cutoff_times["cutoff_time_str"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.strftime("%H:%M")

print("Cutoff execution timestamps (should all be before 09:00 EST):")
cutoff_times[["forecast_date", "cutoff_execution_dt", "cutoff_time_str"]]

Cutoff execution timestamps (should all be before 09:00 EST):


,forecast_date,cutoff_execution_dt,cutoff_time_str
0,2026-02-25,2026-02-25 08:47:27,08:47
1,2026-02-26,2026-02-26 08:47:10,08:47
2,2026-02-27,2026-02-27 08:47:25,08:47
3,2026-02-28,2026-02-28 08:47:10,08:47
4,2026-03-01,2026-03-01 08:47:24,08:47
5,2026-03-02,2026-03-02 08:47:09,08:47
6,2026-03-03,2026-03-03 08:47:32,08:47
7,2026-03-04,2026-03-04 08:47:16,08:47


In [5]:
# Bar chart of cutoff hour-of-day
fig = px.bar(
    cutoff_times,
    x="forecast_date",
    y="cutoff_hour",
    text="cutoff_time_str",
    title="Cutoff Vintage Hour-of-Day (EST) by Forecast Date",
    template="plotly_dark",
)
fig.add_hline(y=9, line_dash="dash", line_color="red", annotation_text="9am EST cutoff")
fig.update_layout(yaxis_title="Hour (EST)", xaxis_title="Forecast Date")
fig.show()

In [6]:
# Lookback coverage: which dates have 12h/24h/48h data
coverage = (
    df.groupby("forecast_date")
    .agg(
        has_12h=("load_mw_12h", lambda x: x.notna().any()),
        has_24h=("load_mw_24h", lambda x: x.notna().any()),
        has_48h=("load_mw_48h", lambda x: x.notna().any()),
        exec_dt_12h=("exec_dt_12h", "first"),
        exec_dt_24h=("exec_dt_24h", "first"),
        exec_dt_48h=("exec_dt_48h", "first"),
    )
    .reset_index()
)
print("Lookback coverage:")
coverage

Lookback coverage:


,forecast_date,has_12h,has_24h,has_48h,exec_dt_12h,exec_dt_24h,exec_dt_48h
0,2026-02-25,False,False,False,NaT,NaT,NaT
1,2026-02-26,True,True,False,2026-02-25 20:17:33,2026-02-25 08:17:27,NaT
2,2026-02-27,True,True,True,2026-02-26 20:47:17,2026-02-26 08:47:10,2026-02-25 08:17:27
3,2026-02-28,True,True,True,2026-02-27 13:47:28,2026-02-27 08:17:24,2026-02-26 08:47:10
4,2026-03-01,True,True,True,2026-02-28 10:47:11,2026-02-28 08:47:10,2026-02-27 08:17:24
5,2026-03-02,True,True,True,2026-03-01 10:47:25,2026-03-01 08:17:24,2026-02-28 08:17:10
6,2026-03-03,True,True,True,2026-03-02 12:47:12,2026-03-02 08:47:09,2026-03-01 08:47:24
7,2026-03-04,True,True,True,2026-03-03 20:47:09,2026-03-03 08:17:32,2026-03-02 08:47:09


## 3. Forecast Evolution — Cutoff vs Lookbacks

In [7]:
# For each region, overlay 48h → 24h → 12h → cutoff for the latest forecast_date
latest_date = df["forecast_date"].max()
df_latest = df[df["forecast_date"] == latest_date].copy()

regions = sorted(df_latest["region"].unique())
fig = make_subplots(
    rows=len(regions), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{r} — {latest_date}" for r in regions],
    vertical_spacing=0.06,
)

colors = {"48h": "#636EFA", "24h": "#EF553B", "12h": "#00CC96", "Cutoff": "#FFA15A"}
dashes = {"48h": "dot", "24h": "dash", "12h": "dashdot", "Cutoff": "solid"}

for i, region in enumerate(regions, 1):
    rdf = df_latest[df_latest["region"] == region].sort_values("hour_ending")

    for label, col, width in [
        ("48h", "load_mw_48h", 1),
        ("24h", "load_mw_24h", 1.5),
        ("12h", "load_mw_12h", 2),
        ("Cutoff", "cutoff_load_mw", 3),
    ]:
        fig.add_trace(
            go.Scatter(
                x=rdf["hour_ending"],
                y=rdf[col],
                name=label,
                line=dict(color=colors[label], dash=dashes[label], width=width),
                showlegend=(i == 1),
            ),
            row=i, col=1,
        )

fig.update_layout(
    height=300 * len(regions),
    template="plotly_dark",
    title_text=f"Forecast Evolution — Latest Date ({latest_date})",
)
fig.update_yaxes(title_text="Load (MW)")
fig.update_xaxes(title_text="Hour Ending", row=len(regions), col=1)
fig.show()

In [8]:
# Same view for all dates — one chart per region, lines per forecast_date
for region in regions:
    rdf = df[df["region"] == region].copy()
    dates = sorted(rdf["forecast_date"].unique())

    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=[f"{region} — Cutoff vs 24h Lookback"],
    )

    for j, d in enumerate(dates):
        ddf = rdf[rdf["forecast_date"] == d].sort_values("hour_ending")
        opacity = 0.3 if d != latest_date else 1.0
        # Cutoff
        fig.add_trace(go.Scatter(
            x=ddf["hour_ending"], y=ddf["cutoff_load_mw"],
            name=f"{d} cutoff",
            line=dict(width=2 if d == latest_date else 1),
            opacity=opacity,
            showlegend=True,
        ))
        # 24h lookback (dashed)
        fig.add_trace(go.Scatter(
            x=ddf["hour_ending"], y=ddf["load_mw_24h"],
            name=f"{d} 24h",
            line=dict(dash="dash", width=1),
            opacity=opacity * 0.6,
            showlegend=False,
        ))

    fig.update_layout(
        height=500, template="plotly_dark",
        title=f"{region} — Cutoff (solid) vs 24h Prior (dashed)",
        xaxis_title="Hour Ending", yaxis_title="Load (MW)",
    )
    fig.show()

## 4. Forecast Deltas — MW Changes at Each Lookback

In [9]:
# Grouped bar charts: delta_12h, delta_24h, delta_48h by hour_ending for latest date
for region in regions:
    rdf = df_latest[df_latest["region"] == region].sort_values("hour_ending")

    fig = go.Figure()
    for col, label, color in [
        ("delta_48h", "48h→Cutoff", "#636EFA"),
        ("delta_24h", "24h→Cutoff", "#EF553B"),
        ("delta_12h", "12h→Cutoff", "#00CC96"),
    ]:
        fig.add_trace(go.Bar(
            x=rdf["hour_ending"], y=rdf[col],
            name=label, marker_color=color,
        ))

    fig.update_layout(
        barmode="group",
        title=f"{region} — Forecast Revision Deltas ({latest_date})",
        xaxis_title="Hour Ending", yaxis_title="Delta (MW)",
        template="plotly_dark", height=400,
    )
    fig.add_hline(y=0, line_color="white", line_width=0.5)
    fig.show()

In [10]:
# Heatmap: 24h delta across all dates × hours for RTO
rto = df[df["region"] == "RTO"].copy()
pivot_24h = rto.pivot_table(
    index="forecast_date", columns="hour_ending", values="delta_24h",
)

fig = px.imshow(
    pivot_24h.values,
    x=[f"HE{int(c)}" for c in pivot_24h.columns],
    y=[str(d) for d in pivot_24h.index],
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="RTO — 24h Forecast Revision (MW) by Date × Hour",
    labels={"color": "Delta MW"},
    template="plotly_dark",
)
fig.update_layout(height=500)
fig.show()

## 5. DA LMP Connection

In [11]:
# Pull DA LMPs for all hubs
da_lmp_query = """
SELECT
  date,
  hour_ending,
  hub,
  lmp_total
FROM dbt_pjm_v1_2026_feb_19.staging_v1_pjm_lmps_hourly
WHERE market = 'da'
  AND date >= CURRENT_DATE - INTERVAL '14 days'
  AND date <= CURRENT_DATE
ORDER BY date, hour_ending, hub
"""
df_lmp = pull_from_db(query=da_lmp_query)
print(f"DA LMPs: {len(df_lmp):,} rows, hubs: {sorted(df_lmp['hub'].unique())}")

# Pull DA cleared load (RTO)
da_load_query = """
SELECT
  date,
  hour_ending,
  region,
  da_load_mw
FROM dbt_pjm_v1_2026_feb_19.staging_v1_pjm_load_da_hourly
WHERE region = 'RTO'
  AND date >= CURRENT_DATE - INTERVAL '14 days'
  AND date <= CURRENT_DATE
ORDER BY date, hour_ending
"""
df_da_load = pull_from_db(query=da_load_query)
print(f"DA Load: {len(df_da_load):,} rows")

DA LMPs: 4,320 rows, hubs: ['AEP GEN HUB', 'AEP-DAYTON HUB', 'ATSI GEN HUB', 'CHICAGO GEN HUB', 'CHICAGO HUB', 'DOMINION HUB', 'EASTERN HUB', 'N ILLINOIS HUB', 'NEW JERSEY HUB', 'OHIO HUB', 'WEST INT HUB', 'WESTERN HUB']
DA Load: 360 rows


In [12]:
# Merge cutoff forecast (RTO) with DA LMPs
rto_cutoff = df[df["region"] == "RTO"][["forecast_date", "hour_ending", "cutoff_load_mw", "delta_24h"]].copy()
rto_cutoff = rto_cutoff.rename(columns={"forecast_date": "date"})

# Merge with LMPs — one row per (date, hour_ending, hub)
merged_lmp = df_lmp.merge(rto_cutoff, on=["date", "hour_ending"], how="inner")
print(f"Merged LMP rows: {len(merged_lmp):,}")

# Merge cutoff forecast with DA cleared load
merged_load = rto_cutoff.merge(
    df_da_load[["date", "hour_ending", "da_load_mw"]],
    on=["date", "hour_ending"],
    how="inner",
)
merged_load["forecast_error_mw"] = merged_load["cutoff_load_mw"] - merged_load["da_load_mw"]
print(f"Merged load rows: {len(merged_load):,}")

Merged LMP rows: 2,304
Merged load rows: 192


In [13]:
# Scatter: 24h forecast revision vs DA LMP, faceted by hub
fig = px.scatter(
    merged_lmp,
    x="delta_24h",
    y="lmp_total",
    color="hub",
    facet_col="hub",
    facet_col_wrap=3,
    title="24h Forecast Revision (MW) vs DA LMP ($/MWh)",
    labels={"delta_24h": "24h Revision (MW)", "lmp_total": "DA LMP ($/MWh)"},
    template="plotly_dark",
    hover_data=["date", "hour_ending"],
)
fig.update_layout(height=600, showlegend=False)
fig.show()

In [14]:
# Forecast accuracy: cutoff forecast vs DA cleared load
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Cutoff Forecast vs DA Cleared Load", "Forecast Error Distribution"],
)

# Scatter + 45-degree line
fig.add_trace(
    go.Scatter(
        x=merged_load["da_load_mw"], y=merged_load["cutoff_load_mw"],
        mode="markers", marker=dict(size=5, opacity=0.6),
        name="Observations",
        text=merged_load.apply(lambda r: f"{r['date']} HE{int(r['hour_ending'])}", axis=1),
    ),
    row=1, col=1,
)
load_range = [
    min(merged_load["da_load_mw"].min(), merged_load["cutoff_load_mw"].min()),
    max(merged_load["da_load_mw"].max(), merged_load["cutoff_load_mw"].max()),
]
fig.add_trace(
    go.Scatter(
        x=load_range, y=load_range,
        mode="lines", line=dict(dash="dash", color="red"),
        name="Perfect forecast",
    ),
    row=1, col=1,
)
fig.update_xaxes(title_text="DA Cleared Load (MW)", row=1, col=1)
fig.update_yaxes(title_text="Cutoff Forecast (MW)", row=1, col=1)

# Error histogram
fig.add_trace(
    go.Histogram(
        x=merged_load["forecast_error_mw"],
        nbinsx=30, name="Error",
        marker_color="#636EFA",
    ),
    row=1, col=2,
)
fig.update_xaxes(title_text="Forecast Error (MW)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.update_layout(height=500, template="plotly_dark", showlegend=True)
fig.show()

In [15]:
# Summary stats: MAE, RMSE, Bias
errors = merged_load["forecast_error_mw"].dropna()
mae = errors.abs().mean()
rmse = np.sqrt((errors ** 2).mean())
bias = errors.mean()

print(f"Cutoff Forecast Accuracy (RTO, vs DA Cleared Load):")
print(f"  MAE:  {mae:,.0f} MW")
print(f"  RMSE: {rmse:,.0f} MW")
print(f"  Bias: {bias:+,.0f} MW (positive = forecast > actual)")
print(f"  N:    {len(errors)} observations")

Cutoff Forecast Accuracy (RTO, vs DA Cleared Load):
  MAE:  4,491 MW
  RMSE: 4,820 MW
  Bias: +4,490 MW (positive = forecast > actual)
  N:    192 observations


## 6. Summary & Next Steps

**Findings:**
- _Fill in after running: Are cutoff vintages consistently before 9am EST?_
- _Fill in: How much do forecasts revise in the last 24h? Is there a systematic direction?_
- _Fill in: Do larger revisions correlate with DA LMP outcomes?_
- _Fill in: How accurate is the cutoff forecast vs DA cleared load?_

**Next steps:**
- If patterns are stable, promote the SQL CTEs to a dbt model
- Consider adding weather data to explain revision patterns
- Extend lookback window as more snapshot data accumulates